# Silero VAD Baseline Metrics on TORGO Pilot

This notebook evaluates Silero VAD performance on the TORGO pilot subset using cached teacher probabilities.

**Key Findings:**
- Silero VAD shows different behavior on dysarthric vs control speech
- Many non-lexical utterances (humming, sustained vowels) are not detected
- AUC cannot be computed with proxy ground truth (all-speech assumption)

**Date Generated:** 2026-03-04

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score, roc_curve

%matplotlib inline

# Set style
plt.style.use('seaborn-v0_8-darkgrid')

## Load Manifest and Teacher Outputs

In [ ]:
# Load manifest
manifest = pd.read_csv('../manifests/torgo_pilot.csv')
print(f"Total utterances: {len(manifest)}")

# Speaker distribution
speaker_counts = manifest['speaker_id'].value_counts()
print("\nSpeaker distribution:")
for speaker, count in speaker_counts.items():
    speaker_type = "Dysarthric" if speaker in ['F01', 'M01'] else "Control"
    print(f"  {speaker} ({speaker_type}): {count} utterances")

In [ ]:
# Load cached teacher probabilities
teacher_dir = Path('../teacher_probs/')

def load_teacher_prob(row):
    """Load teacher probability for a manifest row."""
    speaker = row['speaker_id']
    session = row['session']
    utt_id = row['utt_id']
    
    # Try {speaker}_{session}_{utt_id:04d}.npy pattern
    try:
        utt_num = int(utt_id)
        prob_file = teacher_dir / f"{speaker}_{session}_{utt_num:04d}.npy"
        if prob_file.exists():
            return np.load(prob_file)
    except:
        pass
    
    # Try {speaker}_{utt_id}.npy pattern
    prob_file = teacher_dir / f"{speaker}_{utt_id}.npy"
    if prob_file.exists():
        return np.load(prob_file)
    
    return None

# Load all probabilities
all_probs = []
all_labels = []
speaker_probs = {}
speaker_frames = {}

for _, row in manifest.iterrows():
    probs = load_teacher_prob(row)
    
    if probs is not None:
        speaker = row['speaker_id']
        
        # Proxy ground truth: all frames are speech
        labels = np.ones(len(probs), dtype=int)
        
        all_probs.extend(probs)
        all_labels.extend(labels)
        
        if speaker not in speaker_probs:
            speaker_probs[speaker] = []
            speaker_frames[speaker] = 0
        speaker_probs[speaker].extend(probs)
        speaker_frames[speaker] += len(probs)

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

print(f"Total frames loaded: {len(all_probs):,}")
print(f"\nFrames per speaker:")
for speaker, count in speaker_frames.items():
    print(f"  {speaker}: {count:,} frames")

## Probability Distribution Analysis

In [ ]:
# Plot probability distributions per speaker
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

speakers = ['F01', 'FC01', 'M01']
colors = ['#ff7f0e', '#2ca02c', '#1f77b4']

for i, speaker in enumerate(speakers):
    probs = np.array(speaker_probs[speaker])
    axes[i].hist(probs, bins=50, alpha=0.7, color=colors[i], edgecolor='black')
    axes[i].axvline(x=0.5, color='red', linestyle='--', label='Threshold=0.5')
    axes[i].set_xlabel('Speech Probability')
    axes[i].set_ylabel('Frame Count')
    speaker_type = "Dysarthric" if speaker in ['F01', 'M01'] else "Control"
    axes[i].set_title(f'{speaker} ({speaker_type})\nMean={probs.mean():.3f}, Median={np.median(probs):.3f}')
    axes[i].legend()

plt.tight_layout()
plt.savefig('../local/probability_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Metrics at Threshold 0.5

In [ ]:
threshold = 0.5
predictions = (all_probs >= threshold).astype(int)

# Confusion matrix components
tp = np.sum((predictions == 1) & (all_labels == 1))
fp = np.sum((predictions == 1) & (all_labels == 0))
tn = np.sum((predictions == 0) & (all_labels == 0))
fn = np.sum((predictions == 0) & (all_labels == 1))

# Metrics
miss_rate = fn / (fn + tp) if (fn + tp) > 0 else 0
far = fp / (fp + tn) if (fp + tn) > 0 else 0
accuracy = (tp + tn) / (tp + fp + tn + fn) if (tp + fp + tn + fn) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Threshold: {threshold}")
print(f"\nOverall Metrics:")
print(f"  Miss Rate: {miss_rate:.4f}")
print(f"  False Alarm Rate: {far:.4f}")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  F1 Score: {f1:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TP: {tp:,}  FP: {fp:,}")
print(f"  FN: {fn:,}  TN: {tn:,}")

## Per-Speaker Metrics

In [ ]:
def compute_speaker_metrics(probs, labels, threshold=0.5):
    """Compute metrics for a speaker."""
    probs = np.array(probs)
    labels = np.array(labels)
    predictions = (probs >= threshold).astype(int)
    
    tp = np.sum((predictions == 1) & (labels == 1))
    fp = np.sum((predictions == 1) & (labels == 0))
    tn = np.sum((predictions == 0) & (labels == 0))
    fn = np.sum((predictions == 0) & (labels == 1))
    
    miss_rate = fn / (fn + tp) if (fn + tp) > 0 else 0
    far = fp / (fp + tn) if (fp + tn) > 0 else 0
    accuracy = (tp + tn) / (tp + fp + tn + fn) if (tp + fp + tn + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'miss_rate': miss_rate,
        'far': far,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

# Compute per-speaker metrics
results = []
for speaker in speakers:
    probs = speaker_probs[speaker]
    labels = np.ones(len(probs), dtype=int)
    metrics = compute_speaker_metrics(probs, labels)
    speaker_type = "Dysarthric" if speaker in ['F01', 'M01'] else "Control"
    results.append({
        'Speaker': speaker,
        'Type': speaker_type,
        'Frames': speaker_frames[speaker],
        **metrics
    })

results_df = pd.DataFrame(results)
print("Per-Speaker Metrics @ Threshold 0.5:")
print(results_df.round(4))

In [ ]:
# Visualize per-speaker metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics_to_plot = ['miss_rate', 'accuracy', 'f1']
titles = ['Miss Rate', 'Accuracy', 'F1 Score']

for i, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    values = results_df[metric].values
    colors_bar = ['#ff7f0e' if s in ['F01', 'M01'] else '#2ca02c' for s in results_df['Speaker']]
    
    bars = axes[i].bar(results_df['Speaker'], values, color=colors_bar, edgecolor='black')
    axes[i].set_ylabel(title)
    axes[i].set_ylim(0, 1)
    axes[i].set_title(f'{title} @ Threshold 0.5')
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{value:.3f}', ha='center', va='bottom', fontsize=10)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#ff7f0e', edgecolor='black', label='Dysarthric'),
    Patch(facecolor='#2ca02c', edgecolor='black', label='Control')
]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.02), ncol=2)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('../local/per_speaker_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## Dysarthric vs Control Comparison

In [ ]:
# Group by dysarthric vs control
dysarthric = results_df[results_df['Type'] == 'Dysarthric']
control = results_df[results_df['Type'] == 'Control']

print("DYSARTHric vs CONTROL COMPARISON")
print("="*50)
print(f"\nDysarthric (F01, M01):")
print(f"  Avg Miss Rate: {dysarthric['miss_rate'].mean():.4f}")
print(f"  Avg Accuracy: {dysarthric['accuracy'].mean():.4f}")
print(f"  Avg F1: {dysarthric['f1'].mean():.4f}")

print(f"\nControl (FC01):")
print(f"  Avg Miss Rate: {control['miss_rate'].mean():.4f}")
print(f"  Avg Accuracy: {control['accuracy'].mean():.4f}")
print(f"  Avg F1: {control['f1'].mean():.4f}")

## Key Observations

1. **Counter-intuitive Result**: Control speaker (FC01) has HIGHER miss rate (72.1%) than dysarthric speakers (~52-54%)

2. **Root Cause**: FC01 contains many non-lexical utterances:
   - "[relax your mouth in its normal position]" - silence
   - "[say 'OA' as in cOAt in a very low pitch]" - sustained vowel
   - "[say 'Eee' in a very high pitch]" - humming-like
   - "xxx" - non-speech marker

3. **Silero VAD Limitations**:
   - Not designed to detect non-lexical vocalizations
   - Expects conversational speech patterns
   - Sustained vowels and humming classified as non-speech

4. **Implications for Distillation**:
   - Student model will learn to detect conversational speech
   - May struggle with atypical speech patterns
   - Need to consider data augmentation for non-lexical sounds

## Limitations

1. **Proxy Ground Truth**: All frames assumed speech - true frame-level annotations would improve accuracy
2. **Missing Data**: 2 corrupted F01 files (0067, 0068) and 60 M01 files not cached
3. **Single Threshold**: Metrics computed at 0.5 - optimal threshold may vary by speaker
4. **Pilot Subset**: Results may not generalize to full dataset